# Deliverable 2

In [7]:
# Libs for Deliverable 2
from pathlib import Path
import json
import re


## Task 1

### Part 1 robust file reader

In [6]:
def file_reader(file_path: str | Path) -> tuple[str, str]:
    '''
    Robust file reader that reads a text file using pathlib.
    Tries UTF-8 first and falls back to ISO-8859-1 (Latin-1), the encoding is not hard coded.
    Returns a tuple containing two strings: (file_text_content, encoding_used)
    '''
    path = Path(file_path)

    try:
        return path.read_text(encoding="utf-8"), "utf-8"
    except UnicodeDecodeError:
        return path.read_text(encoding="iso-8859-1"), "iso-8859-1"

### Part 2 regular expression

In [8]:
SALES_PATTERN = re.compile(
    r"^\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] "
    r"\[(?P<register_id>[^\]]+)\] "
    r"item=(?P<item>.+?) "
    r"price=(?P<price>\d+(?:\.\d+)?) "
    r"(?P<currency>[A-Z]{3}) "
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)$"
)


def parse_sales_log(log_text: str) -> list[dict]:
    '''
    Parses raw log text and returns a list of dictionaries, one per valid sales line.
    Noise lines (printer, network, debug messages) do not match the pattern and are skipped.
    '''
    records = []
    for line in log_text.splitlines():
        match = SALES_PATTERN.match(line.strip())
        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)

    return records

# T2 -   Recursive utility functions

In [ ]:
def directory_searcher(directory_path: str | Path) -> list[Path]:
    '''
    Recursively finds all .log files in a nested directory and returns them as a list of Path objects.
    Constraint: does not use Path.glob, Path.rglob or os.walk, only Path.iterdir and recursion.
    '''
    path = Path(directory_path)

    if not path.is_dir():
        print(f"Path {path} is not a directory.")
        return []

    log_files = []
    for item in path.iterdir():
        if item.is_dir():
            log_files.extend(directory_searcher(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files

In [ ]:
for log_file in sorted(directory_searcher(LOG_DIR)):
    _, encoding = file_reader(log_file)
    print(f"{log_file.relative_to(DATA_DIR)}  ({encoding})")

### Part 2: Recursive search for active registers in `store_config.json`

The function does not assume anything about the structure of the file: a dictionary that has `"active": true` and an `"id"` is a register, and every value of a dictionary and every element of a list is searched recursively. This works for any depth and any combination of dicts and lists.

In [ ]:
def get_active_registers(data) -> set[str]:
    '''
    Recursively traverses any combination of nested dicts and lists
    and returns a set of all register IDs where "active" is True.
    '''
    active_ids: set[str] = set()

    if isinstance(data, dict):
        # The dictionary itself is an active register
        if data.get("active") is True and "id" in data:
            active_ids.add(str(data["id"]))

        # Recurse into every value of the dictionary
        for value in data.values():
            active_ids |= get_active_registers(value)

    elif isinstance(data, list):
        # Recurse into every element of the list
        for item in data:
            active_ids |= get_active_registers(item)

    return active_ids


def load_active_registers_from_file(file_path: str | Path) -> set[str]:
    '''Loads store_config.json and returns the active register IDs using get_active_registers.'''
    config_data = json.loads(Path(file_path).read_text(encoding="utf-8"))
    return get_active_registers(config_data)

In [ ]:
# Test on a deeper, irregular structure than the real file
nested_test = {
    "registers": [{"id": "A", "active": True}, {"id": "B", "active": False}],
    "extra": [[{"stores": [{"registers": [{"id": "C", "active": True}]}]}], {"id": "D"}],
}
print("Test structure:", sorted(get_active_registers(nested_test)))   # expected ['A', 'C']
print("store_config.json:", sorted(load_active_registers_from_file(CONFIG_PATH)))

# T3 - Functions for cleaning and analysing datausing NumPy

# T4 - Data analysis using Pandas

# T5 -  Create full workflow